# Franck-Hertz (Mercury) analysis pipeline

Thin driver around `pipeline.py`. Validate on **Run 1** first, then expand to all runs and cross-compare.

Stages: load → denoise → downsample → plot I-vs-V → find maxima → plot maxima → peak spacings.
All knobs live in `pipeline.CONFIG`.

In [ ]:
import importlib
import pipeline
importlib.reload(pipeline)  # re-run this cell after editing pipeline.py
from pipeline import (CONFIG, run_pipeline, load_run, denoise, downsample,
                      plot_iv, find_fh_peaks, plot_overlay,
                      combined_fit, plot_combined_fit, plot_contact_corrected)

# Show inline in the notebook in addition to saving PNGs
%matplotlib inline
CONFIG["show"] = True

## Config toggles
Adjust here, then re-run the cells below. Each stage can be turned off independently.

In [ ]:
# --- noise removal ---
CONFIG["denoise"]["enabled"]       = True
CONFIG["denoise"]["window"]        = 75      # running-average width (samples)
CONFIG["denoise"]["threshold_pct"] = 3.0     # % of I-range deviation to flag
CONFIG["denoise"]["mode"]          = "narrow_spikes"  # narrow_spikes | above | symmetric
CONFIG["denoise"]["max_spike_width"] = 40    # wide enough to remove noise bursts

# --- downsampling ---
CONFIG["downsample"]["enabled"]  = True
CONFIG["downsample"]["method"]   = "random"  # random | stride
CONFIG["downsample"]["fraction"] = 0.10

# --- peak finding / smoothing (also controls the trend curve on the plots) ---
CONFIG["peaks"]["median_window"] = 51      # spike-robust pre-filter (0/1 = off)
CONFIG["peaks"]["smooth_window"] = 401     # rolling-average width for I and V
CONFIG["peaks"]["min_distance"]  = 30
CONFIG["peaks"]["prominence"]    = None      # None -> 10% of smoothed I range

# --- calibration: PER RUN. recorded V * factor = true accelerating V.
#     Run 1 was recorded at 5x; all other runs at 10x. ---
CONFIG["v_calibration"] = {
    "default": 10.0,
    "Run 1":   5.0,
}

# --- peak indexing: n of each run's FIRST detected peak in the V = n*DV + V0 fit.
#     Run 1's sweep started above the 1st maximum, so its first peak is the 2nd. ---
CONFIG["peak_index"] = {
    "default": 1,
    "Run 1":   2,
}

## 1-2. Raw vs denoised (Run 1)

In [ ]:
RUN = "Run 1"
raw = load_run(RUN)
den = denoise(raw)
print(f"raw: {len(raw)}  ->  denoised: {len(den)}  ({100*len(den)/len(raw):.1f}% kept)")
_ = plot_iv(raw, RUN, tag="raw", title=f"{RUN}: raw I vs V")
_ = plot_iv(den, RUN, tag="denoised", title=f"{RUN}: denoised I vs V")

## 3-7. Full pipeline (Run 1)

In [ ]:
res = run_pipeline(RUN, plots=True)
print(res.summary())

## Cross-compare all runs
Run once the Run 1 pipeline looks right. This generates the four graphs for every run into `outputs/<Run N>/` (inline display is suppressed here to avoid 24 figures; re-open the PNGs from the folders).

In [ ]:
import glob, os
import numpy as np

runs = sorted(d for d in glob.glob(os.path.join(CONFIG["data_root"], "*")) if os.path.isdir(d))

# generate per-run folders (4 graphs each) without flooding the notebook with figures
_show = CONFIG["show"]; CONFIG["show"] = False
results = {os.path.basename(r): run_pipeline(r, plots=True) for r in runs}
CONFIG["show"] = _show

print(f"{'run':8s} {'peaks':>5s} {'spacing (recorded V)':>22s} {'spacing (true V)':>18s}")
for name, r in results.items():
    sp = r.spacings_recorded
    mean = sp.mean() if len(sp) else float("nan")
    std  = sp.std(ddof=1) if len(sp) > 1 else 0.0
    cal_mean = r.spacings_calibrated.mean() if len(sp) else float("nan")
    tail = f"{cal_mean:.3f} (x{r.v_factor})" if r.v_factor != 1.0 else "-- (set factor)"
    print(f"{name:8s} {len(r.peaks_v):5d}   {mean:.3f} +/- {std:.3f}      {tail:>18s}")
print(f"\nper-run graphs written to: {CONFIG['save_dir']}/<run>/  (1_raw, 2_denoised, 3_processed, 4_peaks)")

## Combined overlay (all runs, calibrated)
All six calibrated curves on one axis. `normalize=True` scales each curve's current to [0, 1] so the shapes line up despite different current scales (best for comparing peak positions); set `normalize=False` to keep raw current.

In [ ]:
overlay_path = plot_overlay(results, normalize=True)   # set normalize=False for raw current
print(f"overlay written: {overlay_path}  ({len(results)} runs: {', '.join(results)})")

## Combined fit + contact-potential correction
**Shared-slope fit**: one common excitation energy ΔV across all runs, with a separate intercept (contact-potential offset V₀) per run — the right model since runs share ΔV but differ in offset. The **contact-corrected overlay** subtracts each run's V₀ so peaks land on integer multiples of ΔV.

In [ ]:
fit = combined_fit(results)
print(f"shared DV (excitation) = {fit['slope']:.3f} +/- {fit['slope_err']:.3f} V   (Hg literature 4.9 V)")
print(f"RMS residual           = {fit['rms_resid']:.3f} V")
print("per-run contact-potential offset V0:")
for name, off in fit["offsets"].items():
    print(f"   {name}: {off:.3f} V")

plot_combined_fit(results)            # peak voltage vs n, shared-slope fit
plot_contact_corrected(results)       # overlay shifted by V0; peaks on n*DV

## Test: all runs vs. omitting Run 1
Run 1 was taken at a different scope scale and missed its first maximum (its peaks are indexed n=2..5). Compare the shared-slope fit with and without it to see which is cleaner.

In [ ]:
results_no1 = {k: v for k, v in results.items() if k != "Run 1"}

print(f"{'dataset':18s} {'DV (V)':>16s} {'RMS':>7s} {'|DV-4.9|':>9s} {'runs':>5s}")
for label, res in [("ALL (Run1 n=2..5)", results), ("OMIT Run 1", results_no1)]:
    f = combined_fit(res)
    print(f"{label:18s} {f['slope']:.3f} +/- {f['slope_err']:.3f}   "
          f"{f['rms_resid']:.3f}   {abs(f['slope']-4.9):>9.3f}   {len(res):>5d}")

# overlays + fit plots for both (omit-Run-1 versions get a _no_run1 suffix)
plot_overlay(results_no1, tag="overlay_no_run1")
plot_combined_fit(results_no1, tag="_no_run1")
plot_contact_corrected(results_no1, tag="_no_run1")
print("\nwrote *_no_run1 figures to", CONFIG["save_dir"])